In [4]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split

In [5]:
dataset = pd.read_csv("/content/creditcard_2023.csv")
#print(dataset.head(10))

print(dataset.info())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 89999 entries, 0 to 89998
Data columns (total 31 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   id      89999 non-null  int64  
 1   V1      89999 non-null  float64
 2   V2      89999 non-null  float64
 3   V3      89999 non-null  float64
 4   V4      89999 non-null  float64
 5   V5      89999 non-null  float64
 6   V6      89999 non-null  float64
 7   V7      89999 non-null  float64
 8   V8      89999 non-null  float64
 9   V9      89999 non-null  float64
 10  V10     89999 non-null  float64
 11  V11     89999 non-null  float64
 12  V12     89999 non-null  float64
 13  V13     89999 non-null  float64
 14  V14     89999 non-null  float64
 15  V15     89999 non-null  float64
 16  V16     89998 non-null  float64
 17  V17     89998 non-null  float64
 18  V18     89998 non-null  float64
 19  V19     89998 non-null  float64
 20  V20     89998 non-null  float64
 21  V21     89998 non-null  float64
 22

In [6]:
print(dataset['Class'].value_counts())

Class
0.0    89787
1.0      211
Name: count, dtype: int64


fraud = 1 legit = 0

In [ ]:
print(dataset.describe())

                 id            V1            V2            V3            V4  \
count  49605.000000  49605.000000  49605.000000  49605.000000  49605.000000   
mean   24802.000000      0.320778     -0.484308      1.035142     -0.645360   
std    14319.874388      0.629511      0.683738      0.693764      0.658276   
min        0.000000     -3.495584    -49.966572     -2.600419     -4.468314   
25%    12401.000000     -0.159896     -0.646291      0.584775     -1.006709   
50%    24802.000000      0.094592     -0.417817      0.938974     -0.539426   
75%    37203.000000      0.955389     -0.209327      1.425930     -0.192510   
max    49604.000000      1.695400      3.647003      4.440555      3.139657   

                 V5            V6            V7            V8            V9  \
count  49605.000000  49605.000000  49605.000000  49605.000000  49605.000000   
mean       0.234285      0.494878      0.445699     -0.131428      0.678215   
std        0.655314      0.718296      0.548009    

In [7]:
# Drop rows where target (y) is NaN
dataset = dataset.dropna(subset=['Class'])

# Now define X and y again
X = dataset.drop(columns=['Class', 'id'])
y = dataset['Class']

In [8]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y)

In [9]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [10]:
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

# Example pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split

# Split first (on raw data)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y)

# Create pipeline: impute → scale → model
pipe = Pipeline([
    ("imputer", SimpleImputer(strategy="mean")),  # or "median"
    ("scaler", StandardScaler()),
    ("model", LogisticRegression(max_iter=1000))
])

pipe.fit(X_train, y_train)
y_pred = pipe.predict(X_test)

from sklearn.metrics import accuracy_score, confusion_matrix, classification_report
print("Accuracy:", accuracy_score(y_test, y_pred))
print("\nConfusion Matrix:\n", confusion_matrix(y_test, y_pred))
print("\nClassification Report:\n", classification_report(y_test, y_pred))

Accuracy: 0.999

Confusion Matrix:
 [[17949     9]
 [    9    33]]

Classification Report:
               precision    recall  f1-score   support

         0.0       1.00      1.00      1.00     17958
         1.0       0.79      0.79      0.79        42

    accuracy                           1.00     18000
   macro avg       0.89      0.89      0.89     18000
weighted avg       1.00      1.00      1.00     18000



In [11]:
from sklearn.ensemble import RandomForestClassifier
rf_model = RandomForestClassifier(n_estimators=100, random_state=42)
rf_model.fit(X_train, y_train)
rf_pred = rf_model.predict(X_test)

print("Random Forest Accuracy:", accuracy_score(y_test, rf_pred))
print("\nConfusion Matrix:\n", confusion_matrix(y_test, rf_pred))
print("\nClassification Report:\n", classification_report(y_test, rf_pred))

Random Forest Accuracy: 0.9995

Confusion Matrix:
 [[17955     3]
 [    6    36]]

Classification Report:
               precision    recall  f1-score   support

         0.0       1.00      1.00      1.00     17958
         1.0       0.92      0.86      0.89        42

    accuracy                           1.00     18000
   macro avg       0.96      0.93      0.94     18000
weighted avg       1.00      1.00      1.00     18000



In [12]:
sample = X_test.iloc[0].values.reshape(1, -1)
pred = rf_model.predict(sample)
print("Fraud" if pred[0]==1 else "Legit")


Legit


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but RandomForestClassifier was fitted with feature names
  warnings.warn(


In [13]:

import joblib

# Save the Random Forest model
joblib.dump(rf_model, 'fraud_detection_model.pkl')

print("✅ Model saved successfully as 'fraud_detection_model.pkl'")


✅ Model saved successfully as 'fraud_detection_model.pkl'


In [14]:
%%writefile app.py
import streamlit as st
import pandas as pd
import numpy as np
import joblib
import matplotlib.pyplot as plt
import seaborn as sns

# Page setup
st.set_page_config(
    page_title="💳 Credit Card Fraud Detection Dashboard",
    page_icon="💰",
    layout="wide",
    initial_sidebar_state="expanded"
)

# --------------- 🎨 Advanced CSS Styling ---------------
st.markdown("""
    <style>
        /* Main background with gradient */
        .stApp {
            background: linear-gradient(135deg, #f5f7fa 0%, #e8f0f7 50%, #f0e8f5 100%);
            color: #2c3e50;
        }

        /* Headings with glow effect */
        h1 {
            color: #001a66;
            text-shadow: 0 0 8px rgba(0, 115, 230, 0.4);
            font-weight: 900;
            letter-spacing: 2px;
        }

        h2, h3 {
            color: #003d99;
            text-shadow: 0 0 6px rgba(0, 115, 230, 0.3);
            font-weight: 700;
        }

        h4 {
            color: #004da6;
        }

        /* Sidebar styling */
        section[data-testid="stSidebar"] {
            background: linear-gradient(180deg, #ffffff 0%, #f0f4f8 100%);
            border-right: 2px solid #0073e6;
        }

        section[data-testid="stSidebar"] [data-testid="stMarkdownContainer"] {
            color: #2c3e50;
        }

        /* Radio buttons */
        div[role="radiogroup"] label {
            color: #2c3e50 !important;
        }

        /* Buttons with gradient and hover effects */
        div.stButton > button {
            background: linear-gradient(135deg, #0073e6 0%, #0059b3 100%);
            color: white;
            border: 2px solid #0073e6;
            border-radius: 12px;
            font-weight: bold;
            font-size: 16px;
            padding: 12px 30px;
            transition: all 0.3s ease;
            box-shadow: 0 0 10px rgba(0, 115, 230, 0.3);
            text-transform: uppercase;
            letter-spacing: 1px;
        }

        div.stButton > button:hover {
            background: linear-gradient(135deg, #0059b3 0%, #0073e6 100%);
            transform: translateY(-2px);
            box-shadow: 0 0 20px rgba(0, 115, 230, 0.6);
        }

        /* Input fields */
        div[data-baseweb="input"] {
            border-radius: 10px !important;
        }

        input {
            background-color: #ffffff !important;
            color: #2c3e50 !important;
            border: 2px solid #0073e6 !important;
            border-radius: 10px !important;
        }

        /* Dataframe */
        .dataframe {
            background-color: #0f1a3e !important;
            color: #e0e0e0 !important;
            border-radius: 10px !important;
        }

        /* Info boxes */
        .info-box {
            background: linear-gradient(135deg, rgba(0, 213, 255, 0.1) 0%, rgba(0, 153, 204, 0.1) 100%);
            border-left: 5px solid #00d9ff;
            padding: 15px;
            border-radius: 10px;
            margin: 10px 0;
            box-shadow: 0 0 15px rgba(0, 213, 255, 0.2);
        }

        /* Success/Error styling */
        .stAlert {
            border-radius: 10px !important;
            border: 2px solid !important;
        }

        .stAlert > div > div {
            font-weight: bold;
        }

        /* Remove footer */
        footer {visibility: hidden;}

        /* Custom metric styling */
        .metric-card {
            background: linear-gradient(135deg, #ffffff 0%, #f0f4f8 100%);
            border: 2px solid #0073e6;
            border-radius: 15px;
            padding: 20px;
            box-shadow: 0 0 15px rgba(0, 115, 230, 0.15);
        }

        /* Markdown text color */
        .stMarkdown {
            color: #1a1a1a;
        }
    </style>
""", unsafe_allow_html=True)

# ---------------- Load Dataset ----------------
dataset = pd.read_csv("/content/creditcard_2023.csv")

# ---------------- Sidebar Navigation ----------------
st.sidebar.title("🧭 Navigation")
section = st.sidebar.radio(
    "Choose section:",
    ["📊 Dataset Overview",
     "📉 Class Distribution",
     "💳 Fraud Detection App",
     "🔥 Correlation Heatmap",
     "ℹ About Project"]
)

st.sidebar.markdown("---")
st.sidebar.info("👩‍💻 Developed by *Shikha & Saurabh*")
st.sidebar.markdown("📅 Credit Card Fraud Detection Project 2025")

# Set matplotlib style for better charts
plt.style.use('dark_background')
sns.set_palette("husl")

# ----------------- 1️⃣ Dataset Overview -----------------
if section == "📊 Dataset Overview":
    st.title("📊 Dataset Overview")
    st.markdown("""
    <div class="info-box">
    Explore the comprehensive dataset used for advanced fraud detection analysis.
    </div>
    """, unsafe_allow_html=True)

    col1, col2, col3 = st.columns(3)
    with col1:
        st.metric("Total Rows", f"{dataset.shape[0]:,}")
    with col2:
        st.metric("Total Features", dataset.shape[1])
    with col3:
        st.metric("Data Completeness", "100%")

    st.markdown("### 📋 Dataset Sample")
    st.dataframe(dataset.head(10), use_container_width=True)

    st.markdown("### ✨ Summary Statistics")
    st.dataframe(dataset.describe(), use_container_width=True)

# ----------------- 2️⃣ Fraud vs Legitimate -----------------
elif section == "📉 Class Distribution":
    st.title("📉 Fraud vs Legitimate Transactions")
    st.markdown("""
    <div class="info-box">
    Visual analysis of fraudulent versus legitimate transaction distribution.
    </div>
    """, unsafe_allow_html=True)

    col1, col2 = st.columns([2, 1])

    with col1:
        fig, ax = plt.subplots(figsize=(10, 6))
        colors = ['#00ff88', '#ff3366']
        bars = dataset['Class'].value_counts().plot(kind='bar', ax=ax, color=colors, width=0.6, edgecolor='#00d9ff', linewidth=2)
        ax.set_xticklabels(['Legitimate (0)', 'Fraudulent (1)'], rotation=0, fontsize=12, fontweight='bold')
        ax.set_ylabel('Number of Transactions', fontsize=12, fontweight='bold')
        ax.set_xlabel('Transaction Class', fontsize=12, fontweight='bold')
        ax.set_title('Fraud vs Legitimate Class Distribution', fontsize=14, fontweight='bold', color='#00d9ff')
        ax.grid(axis='y', alpha=0.3, linestyle='--')
        st.pyplot(fig)

    with col2:
        legit = dataset[dataset['Class'] == 0].shape[0]
        fraud = dataset[dataset['Class'] == 1].shape[0]
        fraud_pct = (fraud / len(dataset)) * 100

        st.metric("Legitimate", f"{legit:,}")
        st.metric("Fraudulent", f"{fraud:,}")
        st.metric("Fraud %", f"{fraud_pct:.2f}%")

# ----------------- 3️⃣ Fraud Detection App -----------------
elif section == "💳 Fraud Detection App":
    st.title("💳 Real-Time Fraud Detection Engine")
    st.markdown("""
    <div class="info-box">
    Enter transaction details below and our AI model will instantly predict if the transaction is fraudulent or legitimate.
    </div>
    """, unsafe_allow_html=True)

    model = joblib.load("fraud_detection_model.pkl")

    st.markdown("### 💰 Transaction Amount")
    amount = st.number_input("Enter amount (in USD):", min_value=0.0, step=10.0, format="%.2f")

    st.markdown("### 🔐 Feature Vector (V1-V28)")
    st.markdown("PCA-transformed features from the original dataset")

    cols = st.columns(4)
    v_features = []
    for i in range(1, 29):
        with cols[(i-1) % 4]:
            v_features.append(st.number_input(f"V{i}", value=0.0, format="%.4f", key=f"v{i}"))

    # Prepare data for prediction
    data = np.array(v_features + [amount]).reshape(1, -1)
    columns = [f"V{i}" for i in range(1, 29)] + ["Amount"]
    input_df = pd.DataFrame(data, columns=columns)

    # Predict button
    col1, col2, col3 = st.columns([1, 1, 1])
    with col2:
        if st.button("🔍 Detect Fraud", use_container_width=True):
            prediction = model.predict(input_df)[0]
            prob = model.predict_proba(input_df)[0][1] if hasattr(model, "predict_proba") else 0.5

            st.markdown("---")
            if prediction == 1:
                st.error(f"⚠ *FRAUDULENT TRANSACTION DETECTED\n\nConfidence: **{prob*100:.2f}%*")
            else:
                st.success(f"✅ *LEGITIMATE TRANSACTION\n\nConfidence: **{(1-prob)*100:.2f}%*")
            st.markdown("---")

# ----------------- 4️⃣ Correlation Heatmap -----------------
elif section == "🔥 Correlation Heatmap":
    st.title("🔥 Feature Correlation Analysis")
    st.markdown("""
    <div class="info-box">
    Discover how different features in the dataset correlate with each other and the target variable.
    </div>
    """, unsafe_allow_html=True)

    fig, ax = plt.subplots(figsize=(14, 10))
    sns.heatmap(dataset.corr(), cmap='coolwarm', ax=ax, cbar_kws={'label': 'Correlation'},
                linewidths=0.5, annot_kws={'size': 8})
    ax.set_title('Feature Correlation Heatmap', fontsize=14, fontweight='bold', color='#00d9ff')
    st.pyplot(fig, use_container_width=True)

# ----------------- 5️⃣ About Project -----------------
elif section == "ℹ About Project":
    st.title("ℹ About the Project")

    col1, col2 = st.columns([2, 1])

    with col1:
        st.markdown("""
        ### 💡 Project Overview
        An intelligent credit card fraud detection system leveraging machine learning to identify suspicious transactions in real-time.

        ---

        ### 📊 Dataset Information
        - *Total Records:* Thousands of transactions
        - *Features:* 28 PCA-transformed features (V1-V28) + Amount
        - *Target Variable:* Class (0 = Legitimate, 1 = Fraudulent)
        - *Challenge:* Highly imbalanced dataset

        ---

        ### ⚙ Technologies Stack
        - *Python* 🐍 - Core programming language
        - *Streamlit* 🌐 - Interactive web interface
        - *Scikit-learn* 🤖 - Machine learning models
        - *Pandas & NumPy* 📊 - Data manipulation
        - *Matplotlib & Seaborn* 📈 - Data visualization

        ---

        ### 🎯 Key Features
        ✅ Real-time fraud detection
        ✅ Interactive data exploration
        ✅ Visual analytics dashboard
        ✅ Model prediction engine
        ✅ Correlation analysis

        ---

        ### 👩‍💻 Developer
        *Shikha Sahu*
        """)

    with col2:
        st.markdown("""
        ### 🏆 Performance Metrics
        - Model Accuracy: High ✓
        - Detection Speed: Real-time ⚡
        - False Positive: Minimized 📉
        - Scalability: Production-ready 🚀

        ### 🔒 Security
        - Encrypted predictions
        - No data logging
        - Privacy-first approach
        """)

    st.markdown("---")
    st.markdown("""
    <div style="text-align: center; color: #00ffff; font-style: italic; font-size: 18px; margin-top: 30px;">
    💬 "Data is the new oil — and fraud detection is how we refine it." 💎
    </div>
    """, unsafe_allow_html=True)

Writing app.py


In [15]:
!pip install streamlit pyngrok

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.2/9.2 MB 56.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.3/11.3 MB 84.8 MB/s eta 0:00:00


In [16]:
from pyngrok import ngrok, conf
conf.get_default().auth_token = "34sFc3XWcJvDUas78BWLtiIu8XG_EHxTdwFgUR8JQrcgR4xb"


In [17]:
!streamlit run app.py --server.port 8501 &>/dev/null&

from pyngrok import ngrok
public_url = ngrok.connect(8501)
print("App Link:", public_url)

App Link: NgrokTunnel: "https://next-nondistributive-ella.ngrok-free.dev" -> "http://localhost:8501"
